In [ ]:
!pip install langgraph

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 539.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.2/153.2 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.6/50.6 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.5/216.5 kB 5.6 MB/s eta 0:00:00


In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [ ]:
class CrashState(TypedDict):
  input : str
  step1 : str
  step2 : str
  step3 : str

In [ ]:
def step_1(state: CrashState) -> CrashState:
  print("Step1 executed")
  return {"step1": "done", "input": state["input"]}

def step_2(state: CrashState) -> CrashState:
  print("Step2 hanging... now manually interrupt from the notebook")
  time.sleep(100)
  return {"step2": "done"}


def step_3(state: CrashState) -> CrashState:
  print("Step3 executed")
  return {"step3": "done"}

In [ ]:
builder = StateGraph(CrashState)
builder.add_node("step1", step_1)
builder.add_node("step2", step_2)
builder.add_node("step3", step_3)

builder.add_edge(START, "step1")
builder.add_edge("step1", "step2")
builder.add_edge("step2", "step3")
builder.add_edge("step3", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [ ]:
config = {"configurable": {"thread_id": "thread-1"}}

try:
    print("▶️ Running graph: Please manually interrupt during Step 2...")
    graph.invoke({"input":"start"}, config = config)
except KeyboardInterrupt:
    print("⚠️ Interrupted")


▶️ Running graph: Please manually interrupt during Step 2...
Step1 executed
Step2 hanging... now manually interrupt from the notebook
⚠️ Interrupted


In [ ]:
graph.get_state(config=config)

StateSnapshot(values={'input': 'start', 'step1': 'done'}, next=('step2',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f07852b-c8df-6b64-8001-fb9a9e1614ef'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2025-08-13T14:35:19.455817+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f07852b-c8dd-6097-8000-2fa9e2a72914'}}, tasks=(PregelTask(id='2fc4e63c-f837-3ca9-b097-83628f41b392', name='step2', path=('__pregel_pull', 'step2'), error=None, interrupts=(), state=None, result=None),), interrupts=())

In [ ]:
list(graph.get_state_history(config=config))

[StateSnapshot(values={'input': 'start', 'step1': 'done'}, next=('step2',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f07852b-c8df-6b64-8001-fb9a9e1614ef'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2025-08-13T14:35:19.455817+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f07852b-c8dd-6097-8000-2fa9e2a72914'}}, tasks=(PregelTask(id='2fc4e63c-f837-3ca9-b097-83628f41b392', name='step2', path=('__pregel_pull', 'step2'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'input': 'start'}, next=('step1',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f07852b-c8dd-6097-8000-2fa9e2a72914'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2025-08-13T14:35:19.454719+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkp

In [ ]:
final_state = graph.invoke(None, config = config)
print("Final state", final_state)

Step2 hanging... now manually interrupt from the notebook
Step3 executed
Final state {'input': 'start', 'step1': 'done', 'step2': 'done', 'step3': 'done'}


In [18]:
graph.get_state(config=config)

StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done', 'step3': 'done'}, next=(), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f0785aa-12fe-6d1e-8003-9139b2f178de'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2025-08-13T15:31:49.514764+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f0785aa-12fc-6469-8002-68e0a1e897e3'}}, tasks=(), interrupts=())

In [19]:
list(graph.get_state_history(config=config))

[StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done', 'step3': 'done'}, next=(), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f0785aa-12fe-6d1e-8003-9139b2f178de'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2025-08-13T15:31:49.514764+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f0785aa-12fc-6469-8002-68e0a1e897e3'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done'}, next=('step3',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f0785aa-12fc-6469-8002-68e0a1e897e3'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2025-08-13T15:31:49.513670+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f07852b-c8df-6b64-8001-fb9a9e1614ef'}}, tasks=(PregelTask(id='68832fca-8eef-a2c6-